# skill_writer — Demo Live

Usiamo Python per generare skill, command e agent di Claude Code.
Il momento meta: **Claude (API) scrive skill per Claude Code**.

In [1]:
# Setup
import sys, os
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print(f"API key: {os.getenv('ANTHROPIC_API_KEY', 'NOT SET')[:25]}...")

API key: sk-ant-api03-0HlwkUm1gHof...


## 1. Scope Detection

Dove scrive il tool? Dipende dal contesto (global / project / local).

In [2]:
from src.skill_writer import SkillWriter

sw = SkillWriter(scope="project")
print(f"Scope : {sw.scope_name}")
print(f"Path  : {sw.scope_path}")

Scope : project
Path  : /data/dev/demo/ai-dev-v1/.claude


## 2. Creazione Manuale

Scriviamo una skill con contenuto nostro.

In [3]:
result = sw.skill(
    name="summarize",
    description="Riassumi qualsiasi testo o file",
    content="Leggi il contenuto fornito e restituisci un riassunto conciso in italiano, massimo 5 punti.\n\nInput: $ARGUMENTS",
    overwrite=True,
)
print(result)
print("\n--- Contenuto file ---")
print(result.path.read_text())

skill written → /data/dev/demo/ai-dev-v1/.claude/skills/summarize.md

--- Contenuto file ---
---
name: summarize
description: Riassumi qualsiasi testo o file
---

Leggi il contenuto fornito e restituisci un riassunto conciso in italiano, massimo 5 punti.

Input: $ARGUMENTS



## 3. Il Momento Meta: Claude Scrive una Skill per Claude Code

Usiamo l'API Anthropic per generare il contenuto — poi lo scriviamo con SkillWriter.

In [4]:
import anthropic

client = anthropic.Anthropic()

# Claude genera il contenuto della skill
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=800,
    system="Scrivi istruzioni pratiche per una Claude Code skill. Nessun frontmatter, solo il corpo in italiano.",
    messages=[{"role": "user", "content": "Skill: 'security-review' — Analizza il codice Python per vulnerabilità OWASP Top 10"}]
)
content = response.content[0].text
print(content)

# Security Review - Analisi Vulnerabilità OWASP Top 10

## Descrizione
Questa skill analizza codice Python per identificare vulnerabilità comuni secondo la lista OWASP Top 10, fornendo raccomandazioni di correzione specifiche.

## Utilizzo Base
```
Analizza questo codice per vulnerabilità OWASP:
[incolla codice]
```

## Categorie Controllate

### 1. SQL Injection
La skill identifica:
- Query SQL costruite con concatenazione di stringhe
- Uso di `format()` o f-string con input utente
- Mancanza di query parametrizzate

**Esempio vulnerabile:**
```python
query = f"SELECT * FROM users WHERE id = {user_id}"
```

**Correzione consigliata:**
```python
query = "SELECT * FROM users WHERE id = ?"
cursor.execute(query, (user_id,))
```

### 2. Broken Authentication
Controlla:
- Password in plaintext nel codice
- Sessioni non gestite correttamente
- Mancanza di hashing password
- Token hardcoded

### 3. Sensitive Data Exposure
Identifica:
- Credenziali negli script
- Dati sensibili loggati
- Comun

In [5]:
result = sw.skill(
    name="security-review",
    description="Analizza codice Python per vulnerabilità OWASP Top 10",
    content=content,
    overwrite=True,
)
print(result)
print(f"\nToken usati: {response.usage.input_tokens} in / {response.usage.output_tokens} out")

skill written → /data/dev/demo/ai-dev-v1/.claude/skills/security-review.md

Token usati: 63 in / 800 out


## 4. generate=True — Shortcut Integrato

SkillWriter può chiamare l'API direttamente senza boilerplate.

In [6]:
result = sw.skill(
    name="commit-message",
    description="Genera un conventional commit message per le modifiche staged",
    generate=True,
    overwrite=True,
)
print(result)
print("\n--- File generato ---")
print(result.path.read_text())

skill written → /data/dev/demo/ai-dev-v1/.claude/skills/commit-message.md

--- File generato ---
---
name: commit-message
description: Genera un conventional commit message per le modifiche staged
---

<skill-name>
commit-message
</skill-name>

<skill-description>
Generates a conventional commit message for staged changes in Git
</skill-description>

<skill-instructions>

1. Run `git diff --cached --name-only` to get the list of staged files
2. Run `git diff --cached` to view the actual changes staged for commit
3. Analyze the changes to determine:
   - The type of change (feat, fix, docs, style, refactor, perf, test, chore, ci, build)
   - The scope (optional but recommended - the affected component/module)
   - A concise description of what changed (50 characters max for the subject line)
   - Whether a breaking change occurred (requires BREAKING CHANGE: prefix or ! notation)

4. Generate the conventional commit message following this format:
   `<type>(<scope>): <subject>`
   
   Wh

## 5. CLI

Gli stessi risultati dalla riga di comando.

In [7]:
import subprocess
subprocess.run(["python", "-m", "src.skill_writer.cli", "--help"])

/data/dev/demo/llm-ops-v1/.venv/bin/python: Error while finding module specification for 'src.skill_writer.cli' (ModuleNotFoundError: No module named 'src')


CompletedProcess(args=['python', '-m', 'src.skill_writer.cli', '--help'], returncode=1)

In [8]:
subprocess.run([
    "python", "-m", "src.skill_writer.cli",
    "skill", "create", "code-explain",
    "--description", "Spiega cosa fa il codice selezionato in italiano",
    "--generate",
    "--scope", "project",
    "--overwrite",
])

/data/dev/demo/llm-ops-v1/.venv/bin/python: Error while finding module specification for 'src.skill_writer.cli' (ModuleNotFoundError: No module named 'src')


CompletedProcess(args=['python', '-m', 'src.skill_writer.cli', 'skill', 'create', 'code-explain', '--description', 'Spiega cosa fa il codice selezionato in italiano', '--generate', '--scope', 'project', '--overwrite'], returncode=1)

## 6. Inventario

Tutte le skill/command/agent nel progetto corrente.

In [9]:
for kind, paths in sw.list().items():
    print(f"\n{kind}s:")
    for p in paths:
        print(f"  /{p.stem}" if kind == "command" else f"  {p.stem}")


skills:
  commit-message
  security-review
  summarize

commands:

agents:
